# Agente de recomandación de Outfit


Este notebook contiene un 'Agente para la sugerencia de Outfit' diseñado para analizar noticias del tiempo. Su objetivo es predecir cómo te vestirás, prendas favoritas, sueters, etc. Para ello, utiliza técnicas avanzadas de Inteligencia Artificial como modelos de lenguaje grandes (LLMs) y sistemas de recuperación de información (RAG).

## 1. Configuración Inicial y Carga de Modelos

En esta primera sección, prepararemos el entorno para nuestro agente. Esto incluye instalar las herramientas de software necesarias y cargar los modelos de Inteligencia Artificial que usaremos. Estos modelos son la 'mente' de nuestro agente, encargados de entender el lenguaje y generar respuestas.

In [9]:
# Instalar librerías e iniciar sesión en Hugging Face con el token desde Colab Secrets

# 'pip install' es el comando para instalar nuevas herramientas (librerías) en Python.
# Aquí instalamos 'transformers' (para trabajar con modelos de lenguaje), 'peft' y 'accelerate' (para hacer que los modelos sean más eficientes),
# 'trl' (para entrenar modelos de forma más sencilla), 'sentence-transformers' (para convertir texto en números que la IA entienda),
# y 'groq' (para usar un modelo de lenguaje muy potente de Groq).
# '--quiet' hace que la instalación sea menos 'habladora' en pantalla.
!pip install transformers peft accelerate trl sentence-transformers groq --quiet

# Importamos las librerías que acabamos de instalar o que ya están disponibles.
import torch # Librería fundamental para operaciones de IA y el manejo de datos de forma eficiente.
from google.colab import userdata # Nos permite acceder a 'secretos' guardados de forma segura en Google Colab.
from huggingface_hub import login # Para iniciar sesión en Hugging Face, una plataforma donde se comparten modelos de IA.
from transformers import AutoModelForCausalLM, AutoTokenizer # Herramientas para cargar modelos de lenguaje y sus 'tokenizadores' (convertidores de texto).
from transformers.utils import logging # Para manejar los mensajes que nos dan las librerías.

# Asegúrate de tener tu HF_TOKEN en Colab Secrets
# 'userdata.get('HF_TOKEN')' recupera tu token personal de Hugging Face, que es como tu contraseña.
# Esto es necesario para poder descargar y usar algunos modelos.
login(token=userdata.get('hugging'))
print('Sesión de Hugging Face iniciada correctamente.') # Mensaje para confirmar que todo salió bien.

Sesión de Hugging Face iniciada correctamente.


### 1.1 Carga del Modelo Base y Tokenizer

Aquí cargaremos el cerebro de nuestro agente: un 'Modelo de Lenguaje Grande' (LLM). Elegimos un modelo pequeño y eficiente llamado 'TinyLlama'. También cargaremos su 'tokenizer', que es como un traductor que convierte el texto normal en un formato que el modelo puede entender y viceversa.

In [10]:
# Cargar el modelo base y su tokenizer

# Definimos el nombre del modelo que vamos a usar. En este caso, es 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'.
modelo_base_str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# El 'tokenizer' es como un diccionario y un conjunto de reglas para el modelo.
# Le dice al modelo cómo dividir las palabras y convertirlas en números que la IA pueda procesar.
tokenizer = AutoTokenizer.from_pretrained(modelo_base_str)

# Cargamos el modelo de lenguaje en sí. 'AutoModelForCausalLM' es un tipo de modelo que genera texto.
# 'dtype=torch.float16' lo hace más rápido y usa menos memoria.
# 'device_map="auto"' le dice a Python que use el mejor dispositivo disponible (como la tarjeta gráfica) para ejecutar el modelo.
modelo = AutoModelForCausalLM.from_pretrained(modelo_base_str, dtype=torch.float16, device_map="auto")
print("Modelo base TinyLlama cargado.") # Confirmamos que el modelo ha sido cargado.

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base TinyLlama cargado.


### 1.2 Función de Generación de Respuesta (para pruebas iniciales)

Esta es una pequeña función que nos permite interactuar con el modelo de lenguaje que acabamos de cargar. Es útil para hacerle preguntas simples y ver cómo responde antes de integrarlo en nuestro agente más complejo.

In [11]:
def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=60):
    # Esta función toma un modelo, una pregunta y cuántas palabras nuevas debe generar.

    # Preparamos la pregunta en un formato que el modelo entiende, simulando una conversación.
    # 'mensajes' es una lista donde cada elemento es un turno de la conversación (aquí, solo el usuario).
    mensajes = [{"role": "user", "content": pregunta}]

    # El 'tokenizer.apply_chat_template' convierte nuestra lista de mensajes en un formato de texto
    # específico que el modelo TinyLlama espera para las conversaciones.
    # 'tokenize=False' significa que aún no lo convertimos a números, solo lo formateamos como texto.
    # 'add_generation_prompt=True' añade una señal al final para que el modelo sepa que debe empezar a generar su respuesta.
    prompt_formateado = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)

    # Ahora, el texto formateado se convierte en números ('input_ids') que el modelo puede procesar.
    # 'return_tensors="pt"' indica que queremos el resultado como un 'tensor' de PyTorch.
    # '.to(modelo_a_usar.device)' asegura que los números estén en el mismo lugar de la memoria que el modelo (ej. en la tarjeta gráfica).
    entrada = tokenizer(prompt_formateado, return_tensors="pt").to(modelo_a_usar.device)

    # El modelo genera la respuesta.
    # '**entrada' pasa nuestros números de entrada al modelo.
    # 'max_new_tokens' limita la longitud de la respuesta generada.
    # 'do_sample=False' hace que las respuestas sean más predecibles y menos 'creativas'.
    # 'pad_token_id=tokenizer.eos_token_id' ayuda al modelo a saber cuándo terminar una frase.
    # 'no_repeat_ngram_size=3' evita que el modelo repita las mismas tres palabras seguidas.
    salida = modelo_a_usar.generate(
        **entrada, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3,
    )

    # La salida del modelo incluye tanto la pregunta original como la respuesta.
    # Aquí extraemos solo los 'tokens nuevos' (la parte de la respuesta).
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]

    # Finalmente, convertimos los números de la respuesta de nuevo a texto legible y quitamos espacios extra.
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

print("Función 'generar_respuesta' definida.") # Confirmamos que la función está lista para usarse.

Función 'generar_respuesta' definida.


## 2. Base de Conocimiento (RAG) para Indicadores de Clima

In [12]:
from sentence_transformers import SentenceTransformer # Librería para convertir oraciones en 'embeddings' (representaciones numéricas).
import numpy as np # Librería para realizar operaciones matemáticas eficientes con arreglos de números.

# Cargamos un modelo pre-entrenado que es bueno para crear 'embeddings' de texto en varios idiomas.
# Estos embeddings son como coordenadas en un espacio matemático, donde oraciones con significado similar están 'cerca'.
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Esta es nuestra 'base de conocimiento'. Cada frase describe un escenario y su posible impacto en nuestro outfit de hoy
documentos_clima = [
    "Una ola de calor con temperaturas superiores a los 32°C y cielos despejados requiere un outfit fresco (ej. vestido de lino, sandalias, sombrero de ala ancha y lentes de sol).",
    "Un descenso de temperatura por la llegada de un frente frío (ej. entre 5°C y 12°C) requiere un outfit abrigado (ej. suéter de lana, pantalón de mezclilla, abrigo largo, botas y bufanda).",
    "La llegada de lluvias intensas acompañadas de alta humedad requiere un outfit resistente al agua (ej. gabardina impermeable, pantalón de secado rápido, botas impermeables y paraguas).",
    "Un día soleado con temperaturas agradables (ej. entre 20°C y 25°C) permite utilizar un outfit casual y ligero (ej. falda corta, blusa de algodón, bailarinas y bolso pequeño).",
    "La presencia de fuertes ráfagas de viento (ej. entre 40 y 60 km/h) requiere prendas que protejan del viento (ej. pantalón recto, blusa ajustada, chaqueta cortavientos y zapatos cerrados).",
    "Un día nublado con temperaturas moderadas (ej. entre 15°C y 20°C) favorece un outfit de entretiempo (ej. falda midi, cárdigan ligero, medias y mocasines).",
    "Una alta variación térmica durante el día (ej. 10°C por la mañana y 27°C por la tarde) requiere un outfit en capas (ej. camiseta de algodón, pantalón de tiro alto, chaqueta ligera y tenis blancos)."
]

# Convertimos cada documento de nuestra base de conocimiento en un 'embedding' (vector numérico).
# Esto nos permite comparar rápidamente la similitud entre una nueva noticia y nuestro outfit.
embeddings_documentos_clima = modelo_embeddings.encode(documentos_clima)
print("Embeddings de indicadores del clima.")

def buscar_fragmento_clima(noticia):
    # Esta función busca el documento más relevante en nuestra base de conocimiento para una noticia dada.

    # Primero, convertimos la nueva noticia en su 'embedding'.
    embedding_noticia = modelo_embeddings.encode([noticia])

    # Calculamos la 'similitud' entre el embedding de la noticia y todos los embeddings de nuestros documentos.
    # Usamos el producto punto ('np.dot') que es una forma común de medir qué tan 'cerca' están dos vectores.
    similitudes = np.dot(embeddings_documentos_clima, embedding_noticia.T).flatten()

    # Encontramos el índice (la posición) del documento que tiene la mayor similitud.
    indice_mas_similar = np.argmax(similitudes)

    # Devolvemos el texto del documento más similar.
    return documentos_clima[indice_mas_similar]

print("\nEjemplo de fragmento relevante para 'Inflación alta':")
# Probamos la función con una noticia de ejemplo.
print(buscar_fragmento_clima("Reportan inflación más alta de lo esperado en Estados Unidos."))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings de indicadores del clima.

Ejemplo de fragmento relevante para 'Inflación alta':
Una ola de calor con temperaturas superiores a los 32°C y cielos despejados requiere un outfit fresco (ej. vestido de lino, sandalias, sombrero de ala ancha y lentes de sol).


## 3. Ajuste Fino (Fine-Tuning) con LoRA para Sugerencias de Outfits

In [5]:
from datasets import Dataset # Librería para manejar conjuntos de datos de forma eficiente para el entrenamiento de modelos.

def formatear_ejemplo_clima(noticia, recomendacion):
    # Esta función toma una noticia y una recomendación y las convierte en un formato de conversación para el entrenamiento.
    mensajes = [
        {"role": "user", "content": noticia}, # El usuario da la noticia
        {"role": "assistant", "content": recomendacion} # El asistente (modelo) da la recomendación
    ]
    # 'tokenizer.apply_chat_template' convierte estos mensajes en una cadena de texto que el modelo espera.
    return tokenizer.apply_chat_template(mensajes, tokenize=False)

# Pares de ejemplo: Noticia -> Sugerencia de Outfit
# Estos son los datos de entrenamiento. Son ejemplos de cómo una noticia debería llevar a una recomendación específica.
# Estos son los datos de entrenamiento. Son ejemplos de cómo
# una noticia meteorológica debería llevar a una recomendación
# específica de outfit.

pares_clima = [
    ("Se anuncia una ola de calor con temperaturas superiores a los 35°C.",
     "Outfit recomendado: vestido de lino, sandalias, sombrero de ala ancha y lentes de sol."),
    ("La llegada de un frente frío provocará temperaturas inferiores a los 10°C.",
     "Outfit recomendado: suéter de lana, pantalón de mezclilla, abrigo largo, botas y bufanda."),
    ("El pronóstico indica cielos despejados y temperaturas agradables de 23°C.",
     "Outfit recomendado: falda corta, blusa de algodón, bailarinas y bolso pequeño."),
    ("Se esperan lluvias intensas y altos niveles de humedad durante todo el día.",
     "Outfit recomendado: gabardina impermeable, pantalón de secado rápido, botas impermeables y paraguas."),
    ("Las autoridades meteorológicas anuncian ráfagas de viento de hasta 60 km/h.",
     "Outfit recomendado: pantalón recto, blusa ajustada, chaqueta cortavientos y zapatos cerrados."),
    ("Se prevé un día nublado con temperaturas estables de 18°C.",
     "Outfit recomendado: falda midi, cárdigan ligero, medias y mocasines."),
    ("El pronóstico anticipa una mañana fría de 10°C y una tarde cálida de 27°C.",
     "Outfit recomendado: camiseta de algodón, pantalón de tiro alto, chaqueta ligera y tenis blancos."),
    ("Una masa de aire polar provocará temperaturas bajo cero y nevadas.",
     "Outfit recomendado: camiseta térmica, suéter de lana, abrigo acolchado, pantalón térmico, botas de nieve y guantes.")
]

# Convertimos nuestros pares de noticias y recomendaciones en un formato que la librería 'datasets' pueda usar.
# Cada elemento tiene una clave 'texto' que contiene la conversación formateada.
ejemplos_clima = [{"texto": formatear_ejemplo_clima(p, r)} for p, r in pares_clima]
# Creamos el objeto Dataset a partir de nuestra lista de ejemplos.
dataset_clima = Dataset.from_list(ejemplos_clima)

# Mostramos el dataset creado para verificarlo.
dataset_clima

Dataset({
    features: ['texto'],
    num_rows: 8
})

### 3.1 Configuración de LoRA

Aquí configuraremos cómo se aplicará LoRA a nuestro modelo. LoRA (Low-Rank Adaptation) es una técnica para 'ajustar' modelos grandes de manera muy eficiente, sin tener que modificar todas las partes del modelo original. Es como añadir una pequeña capa de aprendizaje extra que se especializa en nuestra tarea.

In [6]:
# Asegurarse de que torchao esté actualizado a una versión compatible con peft
!pip install torchao --upgrade -q

from peft import LoraConfig, get_peft_model # 'LoraConfig' para definir los parámetros de LoRA, 'get_peft_model' para aplicar LoRA.
from transformers import set_seed # Para asegurar que los resultados sean reproducibles (siempre los mismos si se usa la misma semilla).
set_seed(42) # Elegimos un número cualquiera como 'semilla' para la reproducibilidad.

# Creamos un objeto 'LoraConfig' para especificar cómo queremos que LoRA funcione.
config_lora = LoraConfig(
    r=8, # 'r' es el 'rango', un número que controla la complejidad del adaptador LoRA. Valores pequeños son más eficientes.
    lora_alpha=16, # 'lora_alpha' es un factor de escala que ajusta la importancia de los nuevos pesos de LoRA.
    target_modules=["q_proj", "v_proj"], # Especifica qué partes del modelo original se modificarán con LoRA (capas de consulta y valor).
    lora_dropout=0.0, # 'lora_dropout' es una técnica para prevenir el sobreajuste, pero aquí lo desactivamos (0.0).
    task_type="CAUSAL_LM" # Le decimos a LoRA que estamos trabajando con un 'modelo de lenguaje causal' (generación de texto).
)

# Aplicamos la configuración LoRA a nuestro modelo base 'modelo'.
# Esto crea un nuevo modelo 'modelo_lora' que es el modelo base más las pequeñas capas LoRA.
modelo_lora = get_peft_model(modelo, config_lora)

# Imprimimos cuántos parámetros (números ajustables) tiene nuestro modelo después de aplicar LoRA.
# Verás que solo un porcentaje muy pequeño es 'entrenable', lo que demuestra la eficiencia de LoRA.
modelo_lora.print_trainable_parameters()


W1117 03:56:35.730000 2368 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### 3.2 Entrenamiento del Modelo LoRA para Sugerencia de Outfit


In [14]:
from trl import SFTTrainer, SFTConfig # 'SFTTrainer' es una herramienta para entrenar modelos de forma más sencilla.

# Configuramos cómo se realizará el entrenamiento.
config_entrenamiento_clima = SFTConfig(
    output_dir="/content/resultados_pipeline_clima", # Carpeta donde se guardarán los resultados del entrenamiento.
    num_train_epochs=30, # Número de veces que el modelo revisará todo el conjunto de datos de entrenamiento.
    per_device_train_batch_size=5, # Cuántos ejemplos se procesan a la vez durante el entrenamiento.
    learning_rate=2e-4, # El 'ritmo de aprendizaje' del modelo. Qué tan rápido ajusta sus conocimientos.
    logging_steps=1, # Con qué frecuencia se muestran los mensajes de progreso durante el entrenamiento.
    dataset_text_field="texto", # Le decimos al entrenador cuál es la columna de texto en nuestro dataset.
    max_length=128, # Longitud máxima de texto que el modelo procesará en cada ejemplo.
    report_to="none", # Desactiva el envío de reportes a plataformas externas de seguimiento de entrenamiento.
)

# Creamos el 'entrenador' ('SFTTrainer') que se encargará del proceso de ajuste fino.
trainer_clima = SFTTrainer(
    model=modelo_lora, # Usamos el modelo con LoRA ya configurado para entrenar.
    train_dataset=dataset_clima, # Usamos el conjunto de datos que preparamos con noticias y sugerenci de outfit.
    args=config_entrenamiento_clima, # Le pasamos la configuración que definimos arriba.
)

# Iniciamos el entrenamiento del modelo.
resultado_entrenamiento_clima = trainer_clima.train()

# Calculamos y mostramos la 'pérdida' (loss) del modelo.
# La pérdida es un número que indica qué tan bien (o mal) está rindiendo el modelo.
# Una pérdida más baja al final significa que el modelo aprendió mejor.
perdida_inicial_clima = trainer_clima.state.log_history[0]['loss']
perdida_final_clima = trainer_clima.state.log_history[-2]['loss']
print(f"Pérdida al inicio del entrenamiento de clima: {perdida_inicial_clima:.2f}")
print(f"Pérdida final del entrenamiento de clima: {perdida_final_clima:.2f}")
print(f"Reducción: {(1 - perdida_final_clima/perdida_inicial_clima) * 100:.0f}%")

Adding EOS to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/llama/modeling_llama.py:66: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at external/rules_python++pip+torch_tpu_pypi_2_14_0_314_torch/site-packages/torch/include/ATen/core/TensorBody.h:494.)
  return self.weight * hidden_states.to(input_dtype)
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2235: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a 

Step,Training Loss
1,2.758654
2,2.941592
3,2.816941
4,2.407923
5,2.528882
6,2.416983
7,2.224583
8,2.496522
9,2.170541
10,2.106947


Pérdida al inicio del entrenamiento de clima: 2.76
Pérdida final del entrenamiento de clima: 0.82
Reducción: 70%


## 4. Despliegue (Guardado) del Modelo Ajustado

Una vez que hemos ajustado nuestro modelo con LoRA, es importante guardar esos cambios. Al guardar el 'adaptador' de LoRA, podemos reutilizar nuestro modelo especializado más tarde sin tener que volver a entrenarlo, lo que ahorra tiempo y recursos.

Guardamos el adaptador LoRA ajustado para la sugerencia de outfits permitiendo su reutilización sin necesidad de reentrenamiento.

Esto es como guardar la configuración especial que nuestro modelo ha aprendido. Si queremos usar este mismo modelo en otro momento o en otra aplicación, simplemente cargamos el modelo base y luego le 'conectamos' este adaptador LoRA guardado, y estará listo para hacer la sugerencia de outfit en el formato que le enseñamos.

In [15]:
# Guardamos solo las partes de LoRA que hemos ajustado.
# Esto crea una carpeta con los archivos necesarios para cargar este 'conocimiento' específico en el futuro.
modelo_lora.save_pretrained("/content/modelo_ajustado_lora_clima")
print("Adaptador LoRA para sugerencia de outfit guardado en /content/modelo_ajustado_lora_clilma")

Adaptador LoRA para sugerencia de outfit guardado en /content/modelo_ajustado_lora_clilma


## 5. El Pipeline Completo: RAG + Agente de recomandación

Aquí es donde unimos todas las piezas para crear nuestro 'Stylist personal' completo. Combinaremos la capacidad de "buscar" información relevante (RAG) con un modelo de lenguaje muy potente (Groq) para que analice una noticia

Combinamos la recuperación de indicadores del clima con la capacidad de un modelo potente (Groq) para sugerir el outfit.

Este agente funciona de la siguiente manera:
1.  **Recibe una noticia:** El usuario introduce un texto sobre el clima.
2.  **Busca contexto relevante (RAG):** El agente consulta su 'base de conocimiento' (los `documentos_clima`) para encontrar la regla o el escenario más parecido a la noticia.
3.  **Formula una pregunta inteligente:** Con la noticia original y el contexto encontrado, el agente crea una pregunta muy específica para un modelo de lenguaje avanzado (Groq).
4.  **Genera la recomendación final**

In [20]:
from groq import Groq # Importamos la librería para interactuar con la API de Groq.

# Asegúrate de tener tu GROQ_API_KEY en Colab Secrets
# Inicializamos el cliente de Groq. Esto nos permite hacer peticiones a sus modelos de IA.
# 'userdata.get('GROQ_API_KEY')' recupera tu clave API de Groq de forma segura.
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def agente_stylist(noticia):
    # Esta es la función principal de nuestro agente que predice el movimiento del clima.

    # 1. Recupera el fragmento más relevante de nuestros indicadores de clima.

    fragmento_contexto = buscar_fragmento_clima(noticia)

    # 2. Construye el prompt (la instrucción) para el modelo grande, pidiendo un porcentaje CONCRETO.
    # Este es el 'mensaje' que le enviaremos al modelo de Groq. Incluye las instrucciones,
    # el contexto relevante que encontramos y la noticia original.
    prompt_clima = f"""Analiza la siguiente noticia y, basándote SOLO en el contexto de clima proporcionado, decide el outfit de hoy.
Indicadores de clima: {fragmento_contexto}

Noticia: {noticia}

Decisión: El outfit de hoy"""

    # 3. Envía la pregunta al modelo de Groq y obtiene la respuesta.
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", # Aquí usamos un modelo potente de Groq para la generación final.
        messages=[{"role": "user", "content": prompt_clima}], # Le pasamos nuestro prompt como un mensaje del usuario.
        temperature=0.0 # 'temperature=0.0' hace que las respuestas sean más directas y menos creativas, ideal para recomendaciones.
    )
    # Extraemos el texto de la respuesta del modelo y quitamos espacios extra.
    return response.choices[0].message.content.strip()

# Ejemplos de noticias para probar el agente.
# Estas son algunas noticias ficticias que usaremos para ver cómo nuestro agente hace recomendaciones.
noticias_ejemplo = [
    "Una ola de calor azota la Ciudad de México y se alcanzan temperaturas de 37°C.",
    "Se esperan tormentas eléctricas y lluvias durante toda la tarde.",
    "Un frente frío llega a la ciudad provocando temperaturas de 7°C.",
    "La mañana comienza con 12°C, pero se espera una máxima de 28°C.",
    "Se pronostican cielos parcialmente nublados, temperaturas de 22°C y fuertes ráfagas de viento."
]

print("Stylist particular :")
for noticia in noticias_ejemplo:
    print(f"Noticia: {noticia}")
    print(f"Recomendación: {agente_stylist(noticia)}\n")

Stylist particular :
Noticia: Una ola de calor azota la Ciudad de México y se alcanzan temperaturas de 37°C.
Recomendación: **Outfit recomendado para hoy (temperaturas ~37 °C, cielo despejado):**  
- Vestido de lino ligero (o blusa de algodón con pantalones cortos)  
- Sandalias cómodas y transpirables  
- Sombrero de ala ancha para proteger del sol  
- Lentes de sol de buena calidad  
- (Opcional) Protector solar y una botella de agua para mantenerse hidratado.

Noticia: Se esperan tormentas eléctricas y lluvias durante toda la tarde.
Recomendación: **Outfit de hoy (basado en el clima de lluvias intensas y alta humedad):**

- **Capa superior:** Gabardina impermeable de alta resistencia al agua.  
- **Parte inferior:** Pantalón de secado rápido (material sintético o mezcla de poliéster).  
- **Calzado:** Botas impermeables con suela antideslizante.  
- **Accesorios:**  
  - Paraguas resistente al viento.  
  - Gorro o sombrero de ala ancha (opcional, para proteger la cabeza).  
  - Gua